# Entrenamiento y Evaluación de Modelos
**Tesis:** Modelo Predictivo de Resultados — Premier League 2017–2025  
**Input:** `model_dataset_enriched.csv` (generado por `pipeline_features.py`)  
**Modelos evaluados:** Regresión Logística · Random Forest · SVM · Gradient Boosting · MLP · Red Neuronal (PyTorch)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, brier_score_loss, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, calibration_curve)
from sklearn.calibration import CalibratedClassifierCV
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
TARGET = 'target_home_win'

---
## 1. Preparación del Dataset

In [ ]:
df_raw = pd.read_csv('Dataset/model_dataset_enriched.csv')

LEAKAGE = ['season','date','home','away','home_goals','away_goals']
REDUND  = ['home_season_wins','away_season_wins','diff_season_wins',
           'home_season_goals_for','away_season_goals_for',
           'home_season_points','away_season_points',
           'home_roll_win','away_roll_win','diff_roll_win',
           'odds_home','odds_away','odds_draw','day_of_week','away_season_matches']
PLAYER_INDIV = [f'{side}_{c}' for side in ['home','away']
                for c in ['gk_save_pct','gk_cs_pct','gk_ga90','top3_ga_per90',
                          'squad_age','squad_depth','team_tkl_per90','team_int_per90',
                          'team_discipline_per90','avg_on_off']]

df = df_raw.drop(columns=[c for c in LEAKAGE+REDUND+PLAYER_INDIV if c in df_raw.columns]).dropna()
print(f'Dataset: {df.shape}  |  Features: {df.shape[1]-1}')
print(f'Victorias locales (clase 1): {df[TARGET].mean()*100:.1f}%')

X = df.drop(columns=[TARGET])
y = df[TARGET]
split = int(len(df)*0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print(f'Train: {len(X_train)} partidos | Test: {len(X_test)} partidos')
print(f'Balance train: {y_train.mean()*100:.1f}% | Balance test: {y_test.mean()*100:.1f}%')

---
## 2. Validación Cruzada Temporal (TimeSeriesSplit)

Antes de entrenar en el set completo, validamos la estabilidad con 5 folds cronológicos.
Esto simula cómo el modelo hubiera rendido en distintos periodos del pasado.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_configs = [
    ('Reg. Logística', LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced'), True),
    ('Random Forest',  RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=4, random_state=42, n_jobs=-1, class_weight='balanced'), False),
    ('SVM (RBF)',      SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42, class_weight='balanced'), True),
    ('Grad. Boosting', GradientBoostingClassifier(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42), False),
    ('MLP (sklearn)',  MLPClassifier(hidden_layer_sizes=(256,128,64), alpha=1e-3, max_iter=500, random_state=42, early_stopping=True, validation_fraction=0.1), True),
]

cv_results = {}
print('=== TimeSeriesSplit CV — F1-Score ===')
for nombre, clf, scaled in cv_configs:
    X_cv = X_tr_sc if scaled else X_train.values
    scores = cross_val_score(clf, X_cv, y_train, cv=tscv, scoring='f1', n_jobs=-1)
    cv_results[nombre] = scores
    print(f'{nombre:<20}: {scores.mean():.4f} ± {scores.std():.4f}  |  folds: {np.round(scores,3)}')

fig, ax = plt.subplots(figsize=(11, 5))
colores = ['#6baed6','#74c476','#fd8d3c','#9e9ac8','#e07b71']
for (nombre, scores), color in zip(cv_results.items(), colores):
    ax.plot(range(1,6), scores, marker='o', color=color, linewidth=2, label=nombre)
    ax.axhline(scores.mean(), color=color, linestyle='--', alpha=0.3)
ax.set_xlabel('Fold cronológico →')
ax.set_ylabel('F1-Score')
ax.set_title('Estabilidad temporal — TimeSeriesSplit CV (5 folds)', fontsize=12)
ax.legend(loc='upper left', fontsize=9)
ax.set_xticks(range(1,6))
plt.tight_layout()
plt.savefig('plot_07_tscv.png', bbox_inches='tight')
plt.show()

---
## 3. Definición de la Red Neuronal (PyTorch)

In [ ]:
class FootballNet(nn.Module):
    """Red neuronal feed-forward con BatchNorm y Dropout."""
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(128, 64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),   nn.ReLU(),
            nn.Linear(32, 1),    nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

def train_pytorch_model(X_tr, y_tr, X_te, y_te, n_epochs=120, lr=1e-3, batch=64, patience=15):
    X_tr_t = torch.FloatTensor(X_tr)
    y_tr_t = torch.FloatTensor(y_tr.values)
    X_te_t = torch.FloatTensor(X_te)

    model = FootballNet(X_tr.shape[1])
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=40, gamma=0.5)
    crit  = nn.BCELoss()
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=batch, shuffle=True)

    hist_loss = []
    best_loss, patience_cnt, best_state = 1e9, 0, None
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in dl:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        sched.step()
        model.eval()
        with torch.no_grad():
            val_loss = crit(model(X_te_t), torch.FloatTensor(y_te.values)).item()
        hist_loss.append({'epoch':epoch+1,'train':epoch_loss/len(dl),'val':val_loss})
        if val_loss < best_loss - 1e-4:
            best_loss, patience_cnt = val_loss, 0
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'  Early stop en epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        proba = model(X_te_t).numpy()
    return (proba >= 0.5).astype(int), proba, model, pd.DataFrame(hist_loss)

print('Arquitectura de FootballNet:')
print(FootballNet(58))

---
## 4. Entrenamiento de Todos los Modelos

In [ ]:
configs = [
    ('Regresión Logística',  LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced'), True),
    ('Random Forest',        RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=4, random_state=42, n_jobs=-1, class_weight='balanced'), False),
    ('SVM (RBF)',            SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42, class_weight='balanced'), True),
    ('Gradient Boosting',   GradientBoostingClassifier(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42), False),
    ('MLP (sklearn)',        MLPClassifier(hidden_layer_sizes=(256,128,64), alpha=1e-3, max_iter=500, random_state=42, early_stopping=True, validation_fraction=0.1), True),
]

resultados, feat_importance, modelos_fit = [], {}, {}

for nombre, clf, scaled in configs:
    X_tr = X_tr_sc if scaled else X_train.values
    X_te = X_te_sc if scaled else X_test.values
    clf.fit(X_tr, y_train)
    y_pred  = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)[:,1]
    resultados.append({'Modelo':nombre,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision':precision_score(y_test, y_pred, zero_division=0),
        'Recall':   recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':  roc_auc_score(y_test, y_proba),
        'Brier':    brier_score_loss(y_test, y_proba),
    })
    modelos_fit[nombre] = {'clf':clf,'y_pred':y_pred,'y_proba':y_proba}
    if hasattr(clf, 'feature_importances_'):
        feat_importance[nombre] = pd.Series(clf.feature_importances_, index=X_train.columns)
    print(f'[OK] {nombre}')

# PyTorch
print('[..] Red Neuronal PyTorch...')
y_pred_nn, y_proba_nn, nn_model, hist_nn = train_pytorch_model(X_tr_sc, y_train, X_te_sc, y_test)
resultados.append({'Modelo':'Red Neuronal (PyTorch)',
    'Accuracy': accuracy_score(y_test, y_pred_nn),
    'Precision':precision_score(y_test, y_pred_nn, zero_division=0),
    'Recall':   recall_score(y_test, y_pred_nn, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred_nn, zero_division=0),
    'ROC-AUC':  roc_auc_score(y_test, y_proba_nn),
    'Brier':    brier_score_loss(y_test, y_proba_nn),
})
modelos_fit['Red Neuronal (PyTorch)'] = {'y_pred':y_pred_nn,'y_proba':y_proba_nn}
print('[OK] Red Neuronal (PyTorch)')

---
## 5. Tabla Comparativa de Resultados

In [ ]:
df_res = pd.DataFrame(resultados).set_index('Modelo').round(4)
df_res = df_res.sort_values('ROC-AUC', ascending=False)

print('=== TABLA COMPARATIVA FINAL — Test Set ===')
print(df_res.to_string())
print(f'\nBaseline del mercado: 62.97% (accuracy)')

# Gráfico de barras agrupadas
metricas_plot = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(metricas_plot))
n = len(df_res)
w = 0.13
paleta = ['#6baed6','#74c476','#fd8d3c','#9e9ac8','#e07b71','#bcbd22']

for i, (modelo, fila) in enumerate(df_res.iterrows()):
    offset = (i - n/2 + 0.5) * w
    ax.bar(x + offset, fila[metricas_plot].values, w, label=modelo,
           color=paleta[i], edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(metricas_plot, fontsize=11)
ax.set_ylim(0.55, 0.88)
ax.axhline(0.6297, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='Baseline mercado (Acc)')
ax.set_title('Comparación de Modelos — Conjunto de Test', fontsize=13)
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('plot_08_comparacion_modelos.png', bbox_inches='tight')
plt.show()

---
## 6. Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for i, (nombre, cfg) in enumerate(modelos_fit.items()):
    fpr, tpr, _ = roc_curve(y_test, cfg['y_proba'])
    auc = roc_auc_score(y_test, cfg['y_proba'])
    ax.plot(fpr, tpr, color=paleta[i], linewidth=2, label=f'{nombre} (AUC={auc:.4f})')

ax.plot([0,1],[0,1],'k--', linewidth=1, label='Aleatorio (AUC=0.50)')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Curvas ROC — Comparación de Modelos', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('plot_09_curvas_roc.png', bbox_inches='tight')
plt.show()

---
## 7. Curva de Entrenamiento — Red Neuronal PyTorch

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hist_nn['epoch'], hist_nn['train'], color='#6baed6', linewidth=2, label='Loss entrenamiento')
ax.plot(hist_nn['epoch'], hist_nn['val'],   color='#e07b71', linewidth=2, label='Loss validación (test)')
ax.set_xlabel('Época')
ax.set_ylabel('BCE Loss')
ax.set_title('Curva de Entrenamiento — Red Neuronal PyTorch', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('plot_10_training_curve_nn.png', bbox_inches='tight')
plt.show()

---
## 8. Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, (nombre, cfg) in zip(axes.flatten(), modelos_fit.items()):
    cm = confusion_matrix(y_test, cfg['y_pred'])
    auc = roc_auc_score(y_test, cfg['y_proba'])
    f1  = f1_score(y_test, cfg['y_pred'], zero_division=0)
    ConfusionMatrixDisplay(cm, display_labels=['No victoria','Victoria']).plot(
        ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{nombre}\nAUC={auc:.3f} | F1={f1:.3f}', fontsize=9)

plt.suptitle('Matrices de Confusión — Todos los Modelos (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig('plot_11_confusion_matrices.png', bbox_inches='tight')
plt.show()

---
## 9. Calibración de Probabilidades

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Calibración perfecta')

for i, (nombre, cfg) in enumerate(modelos_fit.items()):
    pt, pp = calibration_curve(y_test, cfg['y_proba'], n_bins=10)
    ax.plot(pp, pt, 's-', color=paleta[i], linewidth=2, markersize=5, label=nombre)

ax.set_xlabel('Probabilidad media predicha')
ax.set_ylabel('Fracción de positivos reales')
ax.set_title('Curvas de Calibración — Todos los Modelos', fontsize=12)
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('plot_12_calibracion.png', bbox_inches='tight')
plt.show()

print('Brier Score (menor = mejor calibración):')
for nombre, cfg in modelos_fit.items():
    bs = brier_score_loss(y_test, cfg['y_proba'])
    print(f'  {nombre}: {bs:.4f}')

---
## 10. Importancia de Features — Modelos de Árboles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, nombre in zip(axes, ['Random Forest','Gradient Boosting']):
    imp = feat_importance[nombre].sort_values(ascending=False)

    # Clasificar por grupo
    def grupo(col):
        if any(p in col for p in ['gk_','squad_','tkl_per90','int_per90','discipline','on_off','player_quality','top3_ga']):
            return '#e07b71'   # Jugadores
        elif any(p in col for p in ['odds_','market_']):
            return '#6baed6'   # Mercado
        elif 'streak' in col:
            return '#fd8d3c'   # Rachas
        elif 'roll' in col:
            return '#74c476'   # Forma reciente
        else:
            return '#9e9ac8'   # Otros

    top20 = imp.head(20).sort_values()
    colors_bar = [grupo(c) for c in top20.index]
    top20.plot(kind='barh', color=colors_bar, edgecolor='white', ax=ax)
    ax.set_title(f'Top 20 Features — {nombre}', fontsize=11)
    ax.set_xlabel('Importancia (Gini)')
    for bar in ax.patches:
        ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
                f'{bar.get_width():.4f}', va='center', fontsize=7)

# Leyenda
from matplotlib.patches import Patch
legend_el = [Patch(color='#6baed6',label='Mercado/Odds'), Patch(color='#fd8d3c',label='Rachas'),
             Patch(color='#74c476',label='Forma reciente'), Patch(color='#e07b71',label='Jugadores'),
             Patch(color='#9e9ac8',label='Otros')]
axes[1].legend(handles=legend_el, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('plot_13_feature_importance.png', bbox_inches='tight')
plt.show()

---
## 11. Análisis de Errores — Mejor Modelo

In [ ]:
mejor = df_res['ROC-AUC'].idxmax()
print(f'Mejor modelo por ROC-AUC: {mejor}')

cfg_m = modelos_fit[mejor]
df_err = X_test.copy()
df_err['real']     = y_test.values
df_err['predicho'] = cfg_m['y_pred']
df_err['prob']     = cfg_m['y_proba']
df_err['correcto'] = (df_err['real'] == df_err['predicho']).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribución de probabilidades por acierto/error
for val, color, label in [(1,'#74c476','Acierto'),(0,'#e07b71','Error')]:
    sub = df_err[df_err['correcto']==val]['prob']
    axes[0].hist(sub, bins=25, alpha=0.6, color=color, label=label, density=True)
axes[0].axvline(0.5, color='black', linestyle='--')
axes[0].set_title(f'Probabilidades predichas — {mejor}', fontsize=10)
axes[0].set_xlabel('P(victoria local)')
axes[0].legend()

# Accuracy según certeza del mercado
df_err['confianza_mercado'] = pd.cut(df_err['odds_home_prob'],
    bins=[0,0.3,0.4,0.5,0.6,0.7,1.0], labels=['<30%','30-40%','40-50%','50-60%','60-70%','>70%'])
acc_m = df_err.groupby('confianza_mercado', observed=True)['correcto'].mean()
acc_m.plot(kind='bar', color='#6baed6', ax=axes[1], edgecolor='white')
axes[1].set_title('Accuracy del modelo por certeza del mercado', fontsize=10)
axes[1].set_ylabel('Accuracy')
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5)
axes[1].tick_params(axis='x', rotation=0)

# Accuracy por racha del visitante
df_err['racha_bin'] = pd.cut(df_err['away_unbeaten_streak'],
    bins=[-1,0,2,5,50], labels=['0 (sin racha)','1-2','3-5','6+'])
acc_r = df_err.groupby('racha_bin', observed=True)['correcto'].mean()
acc_r.plot(kind='bar', color='#9e9ac8', ax=axes[2], edgecolor='white')
axes[2].set_title('Accuracy vs racha invicta del visitante', fontsize=10)
axes[2].set_ylabel('Accuracy')
axes[2].axhline(df_err['correcto'].mean(), color='red', linestyle='--', alpha=0.5, label='Media global')
axes[2].legend(fontsize=8)
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('plot_14_analisis_errores.png', bbox_inches='tight')
plt.show()

---
## 12. Resumen Final

In [ ]:
print('='*70)
print('RESUMEN FINAL — TODOS LOS MODELOS')
print('='*70)
print(f'Partidos de test:        {len(y_test)}')
print(f'Baseline mercado:        62.97% accuracy')
print(f'Mejor por ROC-AUC:       {df_res["ROC-AUC"].idxmax()}  (AUC={df_res["ROC-AUC"].max():.4f})')
print(f'Mejor por F1-Score:      {df_res["F1-Score"].idxmax()}  (F1={df_res["F1-Score"].max():.4f})')
print(f'Mejor por Accuracy:      {df_res["Accuracy"].idxmax()}  (Acc={df_res["Accuracy"].max():.4f})')
print(f'Mejor calibración:       {df_res["Brier"].idxmin()}  (Brier={df_res["Brier"].min():.4f})')
print()
print('Tabla ordenada por ROC-AUC:')
print(df_res.to_string())
print()
print('TimeSeriesSplit CV (F1) — Estabilidad:')
for nombre, scores in cv_results.items():
    print(f'  {nombre:<20}: {scores.mean():.4f} ± {scores.std():.4f}')
print('='*70)

df_res.to_csv('resultados_finales.csv')
print('\nResultados guardados: resultados_finales.csv')